In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

In [14]:
df = pd.read_csv("Parkinson.csv")

print(df.isnull().sum())

df["sex"] = df["sex"].astype(int)

X = df.drop(columns=['total_updrs', 'subject'])
y_reg = df['total_updrs']

subject          0
age              0
sex              0
test_time        0
motor_updrs      0
total_updrs      0
jitter           0
jitter_abs       0
jitter_rap       0
jitter_ppq5      0
jitter_ddp       0
shimmer          0
shimmer_db       0
shimmer_apq3     0
shimmer_apq5     0
shimmer_apq11    0
shimmer_dda      0
nhr              0
hnr              0
rpde             0
dfa              0
ppe              0
dtype: int64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

In [16]:
rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20]
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Meilleurs paramètres :", grid.best_params_)
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R² :", r2_score(y_test, y_pred))

Meilleurs paramètres : {'max_depth': 20, 'n_estimators': 200}
MAE : 0.19304680363281868
R² : 0.9986819083760274


In [17]:
importances = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importances)

motor_updrs      0.912050
age              0.048813
test_time        0.018260
sex              0.008708
dfa              0.006267
hnr              0.001174
rpde             0.001040
jitter_abs       0.000694
ppe              0.000515
shimmer_apq5     0.000444
shimmer_dda      0.000367
shimmer_apq3     0.000354
shimmer_apq11    0.000343
jitter_ppq5      0.000246
nhr              0.000172
shimmer          0.000137
jitter           0.000118
jitter_ddp       0.000103
jitter_rap       0.000101
shimmer_db       0.000096
dtype: float64


In [18]:
patient = np.array([[72,5.6431,28.199,34.398,0.00662,0.0000338,
                     0.00401,0.00317,0.01204,0.02565,0.23,
                     0.01438,0.01309,0.01662,0.04314,
                     0.01429,21.64,0.41888,0.54842,0.16006]])

prediction = best_model.predict(patient)
print("Total UPDRS prédit :", prediction[0])

Total UPDRS prédit : 43.57537499999999


c:\Users\darla\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [19]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

pipe_lr.fit(X_train, y_train)

y_pred_lr = pipe_lr.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_lr))
print("R² :", r2_score(y_test, y_pred_lr))

MAE : 2.361985186634709
R² : 0.9099723138562507


In [20]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(random_state=42)

param_grid_gbr = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

grid_gbr = GridSearchCV(
    gbr,
    param_grid_gbr,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_gbr.fit(X_train, y_train)

best_gbr = grid_gbr.best_estimator_

y_pred_gbr = best_gbr.predict(X_test)

print("\n===== Gradient Boosting =====")
print("Meilleurs paramètres :", grid_gbr.best_params_)
print("MAE :", mean_absolute_error(y_test, y_pred_gbr))
print("R² :", r2_score(y_test, y_pred_gbr))


===== Gradient Boosting =====
Meilleurs paramètres : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
MAE : 0.40916334631259393
R² : 0.9970244721500953
